# 02 - External Data Extraction (Per-API, Chunked)

In [ ]:
from snowflake.snowpark.context import get_active_session
import time
import os

session = get_active_session()

requirements = """
rioxarray==0.17.0
pystac==1.11.0
pystac_client==0.9.0
planetary_computer==1.0.0
odc-stac==0.3.10
shapely==2.1.1
rasterio==1.4.3
tqdm==4.66.5
pandas==2.3.0
xarray==2025.3.1
matplotlib==3.10.3
geopandas==1.1.1
seaborn==0.13.2
scikit-learn==1.5.2
netCDF4==1.7.2
adlfs==2025.8.0
zarr==2.17.2
dask==2024.10.0
xarray[complete]
numcodecs==0.12.1
pyarrow
pyproj
"""

with open('/tmp/requirements.txt', 'w') as f:
    f.write(requirements)

print("File requirements.txt berhasil dibuat di /tmp")

session.sql("""
    PUT file:///tmp/requirements.txt
    snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live/
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("File ter-upload ke environment Snowflake!")

start_time = time.time()
print("Start Install Package ...")

!pip install uv
!uv pip install -r /tmp/requirements.txt

elapsed_time = time.time() - start_time
print(f"\n✓ Instalasi Package Selesai 100%!")
print(f"  Waktu yang dihabiskan: {elapsed_time:.1f} detik ({elapsed_time/60:.1f} menit)")

In [ ]:
import time
import os
import requests
import pandas as pd
import numpy as np
import pyproj
import rasterio
import math
import geopandas as gpd
import pystac_client
import planetary_computer
import warnings
from rasterio.mask import mask
from rasterio.windows import from_bounds
from shapely.geometry import Point
from rasterio.sample import sample_gen
from shapely.ops import transform
from typing import Dict, Any, Optional, List, Tuple
from datetime import timedelta
from IPython.display import clear_output
from functools import lru_cache
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE DATABASE SNOWFLAKE_LEARNING_DB").collect()
session.sql("USE SCHEMA PUBLIC").collect()
data_stage_path = "@EXTERNAL_DATA_STAGE"

# API helper with retry & exponential backoff 
def _execute_api_get_request(url: str, params: Dict[str, Any], max_retries: int = 5, backoff_factor: float = 1.0, timeout: int = 60) -> Optional[Dict]:
    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except requests.exceptions.HTTPError as e:
            if e.response is not None and e.response.status_code == 429:
                wait_time = backoff_factor * (2 ** attempt)
                print(f"Rate limit hit (429), retrying in {wait_time:.1f}s... (attempt {attempt+1}/{max_retries})")
                time.sleep(wait_time)
                continue
            else:
                print(f"HTTP Error: {e}")
                return None
        except requests.exceptions.RequestException as e:
            wait_time = backoff_factor * (2 ** attempt)
            print(f"Request error: {e}, retrying in {wait_time:.1f}s...")
            time.sleep(wait_time)
            continue
    return None

# Menghasilkan 5 koordinat (Pusat, Utara, Selatan, Timur, Barat) untuk simulasi Buffer
def get_cross_points(lat: float, lon: float, offset_km: float = 1.0) -> List[Tuple[float, float]]:
    # Aproksimasi: 1 derajat Lintang ~ 111 km. 1 derajat Bujur ~ 111 * cos(lat) km.
    lat_offset = offset_km / 111.0
    lon_offset = offset_km / (111.0 * np.cos(np.radians(lat)))
    
    return [
        (lat, lon),                                  # Pusat
        (lat + lat_offset, lon),                     # Utara
        (lat - lat_offset, lon),                     # Selatan
        (lat, lon + lon_offset),                     # Timur
        (lat, lon - lon_offset)                      # Barat
    ]

# ------ Elevation (OpenTopoData) ------
@lru_cache(maxsize=512)
def fetch_elevation_and_slope(lat: float, lon: float) -> Dict[str, float]:
    """Mengambil 5 titik sekaligus untuk menghitung Slope dan Elevasi rata-rata (O(1) request)."""
    points = get_cross_points(lat, lon, offset_km=1.0)
    loc_string = "|".join([f"{p[0]},{p[1]}" for p in points])
    
    data = _execute_api_get_request("https://api.opentopodata.org/v1/aster30m", {"locations": loc_string})
    
    results = {"dem_elev_mean_1km": 0.0, "dem_slope_1km": 0.0}
    if data and "results" in data:
        elevations = [res["elevation"] for res in data["results"] if res["elevation"] is not None]
        if elevations:
            elev_max = max(elevations)
            elev_min = min(elevations)
            # Slope = (Max Elev - Min Elev) / Jarak Horizontal (2 km bentang silang)
            results["dem_elev_mean_1km"] = sum(elevations) / len(elevations)
            results["dem_slope_1km"] = (elev_max - elev_min) / 2000.0 
    return results

# ------ Historical Weather (Open-Meteo) ------
@lru_cache(maxsize=2048)
def extract_weather_vectorized(df: pd.DataFrame, lat_col: str = 'Latitude', lon_col: str = 'Longitude', date_col: str = 'Sample Date') -> pd.DataFrame:
    """
    Sangat Cepat! Menarik rentang waktu penuh per stasiun unik, menghitung rolling 7 hari di Pandas,
    lalu menggabungkannya kembali ke DataFrame asli.
    """
    df[date_col] = pd.to_datetime(df[date_col], format='mixed', dayfirst=False)
    unique_stations = df[[lat_col, lon_col]].drop_duplicates()
    
    all_station_weather = []
    base_url = "https://archive-api.open-meteo.com/v1/archive"
    
    for _, station in unique_stations.iterrows():
        lat, lon = station[lat_col], station[lon_col]
        station_dates = df[(df[lat_col] == lat) & (df[lon_col] == lon)][date_col]
        min_date = (station_dates.min() - timedelta(days=7)).strftime('%Y-%m-%d')
        max_date = station_dates.max().strftime('%Y-%m-%d')
        
        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": min_date,
            "end_date": max_date,
            "daily": ["precipitation_sum", "temperature_2m_max", "windspeed_10m_max"],
            "timezone": "Africa/Johannesburg"
        }
        data = _execute_api_get_request(base_url, params)
        
        if data and "daily" in data:
            weather_df = pd.DataFrame(data["daily"])
            weather_df["time"] = pd.to_datetime(weather_df["time"])
            weather_df[lat_col] = lat
            weather_df[lon_col] = lon
            weather_df["weather_precip_7d_sum"] = weather_df["precipitation_sum"].rolling(window=7, min_periods=1).sum()
            weather_df["weather_temp_7d_max"] = weather_df["temperature_2m_max"].rolling(window=7, min_periods=1).max()
            weather_df["weather_wind_7d_mean"] = weather_df["windspeed_10m_max"].rolling(window=7, min_periods=1).mean()
            
            all_station_weather.append(weather_df)
    master_weather_df = pd.concat(all_station_weather, ignore_index=True)
    final_df = pd.merge(
        df, 
        master_weather_df[[lat_col, lon_col, 'time', 'weather_precip_7d_sum', 'weather_temp_7d_max', 'weather_wind_7d_mean']], 
        left_on=[lat_col, lon_col, date_col], 
        right_on=[lat_col, lon_col, 'time'], 
        how='left'
    ).drop(columns=['time'])
    return final_df

# ------ Soil Properties (ISRIC SoilGrids) ------
@lru_cache(maxsize=2048)
def fetch_soilgrids_properties(latitude: float, longitude: float) -> Dict[str, float]:
    base_url = "https://rest.isric.org/soilgrids/v2.0/properties/query"
    properties = ["phh2o", "clay", "sand", "silt", "organic_carbon", "cec"]
    results = {}
    for prop in properties:
        params = {"lat": latitude, "lon": longitude, "property": prop, "depth": "0-5cm", "value": "mean"}
        data = _execute_api_get_request(base_url, params)
        if data and "properties" in data and "layers" in data["properties"]:
            try:
                mean_val = data["properties"]["layers"][0]["depths"][0]["values"]["mean"]
                results[f"soil_{prop}_mean_0_5cm"] = float(mean_val) if mean_val is not None else 0.0
            except (KeyError, IndexError):
                results[f"soil_{prop}_mean_0_5cm"] = 0.0
        else:
            results[f"soil_{prop}_mean_0_5cm"] = 0.0
    return results

# OSM Pollution Proximity (Overpass)
def calculate_haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Menghitung jarak dalam meter antara dua koordinat."""
    R = 6371000  # Radius bumi dalam meter
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)
    
    a = math.sin(delta_phi / 2.0) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda / 2.0) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    
    return R * c

@lru_cache(maxsize=512)
def fetch_osm_advanced(latitude: float, longitude: float) -> Dict[str, float]:
    """Menarik data OSM dalam radius 5km, menghitung jarak terdekat dan kepadatan buffer."""
    base_url = "http://overpass-api.de/api/interpreter"
    max_radius = 5000
    query = f"""
    [out:json][timeout:50];
    (
      node(around:{max_radius},{latitude},{longitude})["man_made"="mine"];
      way(around:{max_radius},{latitude},{longitude})["man_made"="mine"];
      node(around:{max_radius},{latitude},{longitude})["man_made"="wastewater_plant"];
      way(around:{max_radius},{latitude},{longitude})["man_made"="wastewater_plant"];
      node(around:{max_radius},{latitude},{longitude})["landuse"="farmland"];
      way(around:{max_radius},{latitude},{longitude})["landuse"="farmland"];
    );
    out center;
    """
    results = {
        "osm_mine_count_1km": 0, "osm_mine_count_5km": 0, "osm_dist_nearest_mine": 5000.0,
        "osm_wastewater_count_1km": 0, "osm_wastewater_count_5km": 0, "osm_dist_nearest_wastewater": 5000.0,
        "osm_farm_count_1km": 0, "osm_farm_count_5km": 0, "osm_dist_nearest_farm": 5000.0
    }
    data = _execute_api_get_request(base_url, {"data": query})
    time.sleep(2)
    
    if data and "elements" in data:
        for el in data["elements"]:
            if el["type"] == "node":
                el_lat, el_lon = el["lat"], el["lon"]
            elif el["type"] == "way" and "center" in el:
                el_lat, el_lon = el["center"]["lat"], el["center"]["lon"]
            else:
                continue
        
            dist_m = calculate_haversine_distance(latitude, longitude, el_lat, el_lon)
            tags = el.get("tags", {})
            cat = None
            if tags.get("man_made") == "mine": cat = "mine"
            elif tags.get("man_made") == "wastewater_plant": cat = "wastewater"
            elif tags.get("landuse") == "farmland": cat = "farm"
            
            if cat:
                if dist_m < results[f"osm_dist_nearest_{cat}"]:
                    results[f"osm_dist_nearest_{cat}"] = dist_m
                results[f"osm_{cat}_count_5km"] += 1
                if dist_m <= 1000:
                    results[f"osm_{cat}_count_1km"] += 1
    return results

def fetch_terraclimate_month(start_date: str, end_date: str, coords: List[Tuple[float, float]], target_assets: List[str]) -> Dict[Tuple[float, float], Dict[str, Optional[float]]]:
    """
    Menarik data TerraClimate untuk rentang waktu spesifik (1 bulan) dan sekumpulan koordinat.
    Mengembalikan dictionary dengan mapping: (lon, lat) -> {asset_name: value}
    """
    results = {coord: {asset: None for asset in target_assets} for coord in coords}
    
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=planetary_computer.sign_inplace,
    )
    search = catalog.search(
        collections=["terraclimate"],
        datetime=f"{start_date}/{end_date}",
    )
    items = list(search.items())
    
    if not items:
        print(f"Peringatan: Tidak ada data TerraClimate pada periode {start_date} hingga {end_date}")
        return results
        
    item = items[0] 
    for asset in target_assets:
        if asset in item.assets:
            asset_href = item.assets[asset].href
            try:
                with rasterio.open(asset_href) as src:
                    # sample_gen mengekstrak nilai di koordinat secara efisien
                    sampled_values = list(sample_gen(src, coords))
                    
                    for i, coord in enumerate(coords):
                        val = sampled_values[i][0]
                        # Hanya simpan nilai jika bukan NoData
                        if val != src.nodata:
                            results[coord][asset] = float(val)
            except Exception as e:
                print(f"Error membaca asset '{asset}' via Rasterio: {e}")
                
    return results
def fetch_landsat_bbox(latitude: float, longitude: float, date_str: str) -> dict:
    """
    Menarik citra Landsat (Buffer 1km), mencari citra terbersih dalam jendela waktu +/- 15 hari,
    membersihkan awan, dan mengekstrak fitur turunan (NDVI, LST, dll).
    """
    delta = 0.0045
    bbox = [longitude - delta, latitude - delta, longitude + delta, latitude + delta]
    
    target_date = pd.to_datetime(date_str, format='mixed', dayfirst=False)
    start_date = (target_date - timedelta(days=15)).strftime('%Y-%m-%d')
    end_date = (target_date + timedelta(days=15)).strftime('%Y-%m-%d')
    
    results = {
        'landsat_ndvi_mean_1km': np.nan, 'landsat_ndvi_std_1km': np.nan,
        'landsat_mndwi_mean_1km': np.nan, 'landsat_lst_mean_1km': np.nan,
        'landsat_nir_mean_1km': np.nan, 'landsat_swir16_mean_1km': np.nan
    }
    
    # 3. Kueri ke STAC API
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=planetary_computer.sign_inplace,
    )
    
    search = catalog.search(
        collections=["landsat-c2-l2"],
        bbox=bbox,
        datetime=f"{start_date}/{end_date}",
        query={"eo:cloud_cover": {"lt": 40}}
    )
    items = list(search.items())
    if not items:
        return results
        
    items.sort(key=lambda x: x.properties.get("eo:cloud_cover", 100))
    best_item = items[0]
    
    bands_to_fetch = ['nir08', 'red', 'green', 'swir16', 'lwir11', 'qa_pixel']
    arrays = {}
    
    try:
        for band in bands_to_fetch:
            href = best_item.assets[band].href
            with rasterio.open(href) as src:
                window = from_bounds(*bbox, transform=src.transform)
                arrays[band] = src.read(1, window=window).astype(float)
    except Exception as e:
        print(f"Error membaca raster Landsat: {e}")
        return results

    qa = arrays['qa_pixel']
    cloud_shadow_bit = (1 << 4)
    cloud_bit = (1 << 3)
    dilated_cloud_bit = (1 << 1)
    
    valid_mask = (
        (qa.astype(int) & cloud_shadow_bit == 0) &
        (qa.astype(int) & cloud_bit == 0) &
        (qa.astype(int) & dilated_cloud_bit == 0) &
        (qa > 0)
    )
    for band in ['nir08', 'red', 'green', 'swir16', 'lwir11']:
        arrays[band] = np.where(valid_mask, arrays[band], np.nan)
        
    nir = arrays['nir08']
    red = arrays['red']
    green = arrays['green']
    swir16 = arrays['swir16']
    lst_raw = arrays['lwir11']
    
    ndvi = (nir - red) / (nir + red + 1e-8)
    mndwi = (green - swir16) / (green + swir16 + 1e-8)
    lst_kelvin = lst_raw * 0.00341802 + 149.0

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning) # Abaikan peringatan jika semua piksel adalah NaN
        results['landsat_ndvi_mean_1km'] = np.nanmean(ndvi)
        results['landsat_ndvi_std_1km'] = np.nanstd(ndvi)
        results['landsat_mndwi_mean_1km'] = np.nanmean(mndwi)
        results['landsat_lst_mean_1km'] = np.nanmean(lst_kelvin)
        results['landsat_nir_mean_1km'] = np.nanmean(nir)
        results['landsat_swir16_mean_1km'] = np.nanmean(swir16)
        
    return results

def fetch_sentinel2_bbox(latitude: float, longitude: float, date_str: str) -> dict:
    """
    Menarik citra Sentinel-2 (Resolusi 10m, Buffer 1km), mencari citra terbersih dalam +/- 15 hari,
    melakukan klasifikasi awan SCL, dan mengekstrak indeks turunan.
    """
    delta = 0.0045 # Menghasilkan BBox sekitar 1x1 km
    bbox = [longitude - delta, latitude - delta, longitude + delta, latitude + delta]
    
    target_date = pd.to_datetime(date_str, format='mixed', dayfirst=False)
    start_date = (target_date - timedelta(days=15)).strftime('%Y-%m-%d')
    end_date = (target_date + timedelta(days=15)).strftime('%Y-%m-%d')
    
    results = {
        's2_ndvi_mean_1km': np.nan, 's2_ndvi_std_1km': np.nan,
        's2_mndwi_mean_1km': np.nan, 's2_nir_mean_1km': np.nan
    }
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=planetary_computer.sign_inplace,
    )
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=bbox,
        datetime=f"{start_date}/{end_date}",
        query={"eo:cloud_cover": {"lt": 40}}
    )
    
    items = list(search.items())
    if not items:
        return results
        
    items.sort(key=lambda x: x.properties.get("eo:cloud_cover", 100))
    best_item = items[0]
    
    # PERBEDAAN 2: Nama Band (SCL digunakan untuk masking awan)
    bands_to_fetch = ['B08', 'B04', 'B03', 'B11', 'SCL']
    arrays = {}
    
    try:
        for band in bands_to_fetch:
            href = best_item.assets[band].href
            with rasterio.open(href) as src:
                window = from_bounds(*bbox, transform=src.transform)
                arrays[band] = src.read(1, window=window).astype(float)
    except Exception as e:
        print(f"Error membaca raster Sentinel-2: {e}")
        return results
    scl = arrays['SCL']
    valid_mask = np.isin(scl, [4, 5, 6, 7])
    
    for band in ['B08', 'B04', 'B03', 'B11']:
        arrays[band] = np.where(valid_mask, arrays[band], np.nan)
        
    nir = arrays['B08']
    red = arrays['B04']
    green = arrays['B03']
    swir = arrays['B11']
    
    ndvi = (nir - red) / (nir + red + 1e-8)
    mndwi = (green - swir) / (green + swir + 1e-8)
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        results['s2_ndvi_mean_1km'] = np.nanmean(ndvi)
        results['s2_ndvi_std_1km'] = np.nanstd(ndvi)
        results['s2_mndwi_mean_1km'] = np.nanmean(mndwi)
        results['s2_nir_mean_1km'] = np.nanmean(nir)
        
    return results

# ------ Static Raster / Vector Layers ------
@lru_cache(maxsize=128)
def _get_loaded_static_layer(layer: str):
    base = "/tmp/"
    try:
        if layer == "worldpop":   return rasterio.open(base + "zaf_pop_2025_CN_100m_R2025A_v1.tif")
        if layer == "sanlc2022":  return rasterio.open(base + "SA_NLC_2022_ALBERS.tif")
        if layer == "sanlc2020":  return rasterio.open(base + "SA_NLC_2020_ALBERS.tif")
        if layer == "hydroatlas": return gpd.read_parquet(base + "BasinATLAS_v10_lev12.parquet")
        if layer == "riveratlas": return gpd.read_parquet(base + "RiverATLAS_Data_v10.parquet")
    except Exception as e:
        print(f"Load error {layer}: {e}")
        return None

def fetch_worldpop_features(latitude: float, longitude: float) -> Dict[str, float]:
    """
    Ekstrak fitur turunan WorldPop (Total, Rata-rata, Max) untuk area mikro (1km) dan makro (5km).
    Mengembalikan 6 fitur prediktif.
    """
    raster_source = _get_loaded_static_layer("worldpop")
    if raster_source is None:
        return {}

    results = {
        'worldpop_sum_1km': 0.0, 'worldpop_mean_1km': 0.0, 'worldpop_max_1km': 0.0,
        'worldpop_sum_5km': 0.0, 'worldpop_mean_5km': 0.0, 'worldpop_max_5km': 0.0
    }

    try:
        raster_crs = raster_source.crs
        if raster_crs and not raster_crs.is_geographic:
            transformer = pyproj.Transformer.from_crs("EPSG:4326", raster_crs, always_xy=True)
            x_proj, y_proj = transformer.transform(longitude, latitude)
            point_projected = Point(x_proj, y_proj)
        else:
            point_projected = Point(longitude, latitude)

        for radius_km in [1, 5]:
            radius_m = radius_km * 1000
            
            if raster_crs and not raster_crs.is_geographic:
                buffered_area = point_projected.buffer(radius_m)
            else:
                degree_buffer = radius_m / 111320.0
                buffered_area = point_projected.buffer(degree_buffer)

            raster_nodata = raster_source.nodata
            masked_image, _ = mask(
                raster_source, [buffered_area], crop=True,
                nodata=raster_nodata if raster_nodata is not None else -9999
            )

            pixel_data = masked_image[0].flatten()
            
            # Asumsi standar WorldPop: nilai < 0 adalah no data/air
            if raster_nodata is not None:
                valid_pixels = pixel_data[(pixel_data != raster_nodata) & (pixel_data >= 0)]
            else:
                valid_pixels = pixel_data[pixel_data >= 0]

            if valid_pixels.size > 0:
                results[f'worldpop_sum_{radius_km}km'] = float(valid_pixels.sum())
                results[f'worldpop_mean_{radius_km}km'] = float(valid_pixels.mean())
                results[f'worldpop_max_{radius_km}km'] = float(valid_pixels.max())

        return results

    except Exception as e:
        print(f"Error ekstraksi WorldPop di {latitude},{longitude}: {e}")
        return results

@lru_cache(maxsize=2)
def _load_raster_attribute_table(layer_identifier: str) -> Dict[float, str]:
    base_directory = "/tmp/"
    file_mapping = {
        "sanlc2022": "SA_NLC_2022_ALBERS.tif.vat.dbf",
        "sanlc2020": "SA_NLC_2020_ALBERS.tif.vat.dbf"
    }
    target_file = file_mapping.get(layer_identifier)
    if not target_file:
        return {}
    try:
        dbf_dataframe = gpd.read_file(f"{base_directory}{target_file}")
        value_column = 'Value' if 'Value' in dbf_dataframe.columns else dbf_dataframe.columns[0]
        class_column = 'LC_Class' if 'LC_Class' in dbf_dataframe.columns else dbf_dataframe.columns[1]
        return dict(zip(dbf_dataframe[value_column], dbf_dataframe[class_column]))
    except Exception as initialization_error:
        print(f"Failed to load attribute table for {layer_identifier}: {initialization_error}")
        return {}

def fetch_sanlc_proportions(latitude: float, longitude: float, layer_identifier: str, buffer_radius_meters: int = 1000) -> Dict[str, float]:
    """
    Menghitung persentase luasan setiap tipe tutupan lahan dalam radius buffer.
    Mengembalikan dictionary: { 'sanlc2022_pct_Urban': 0.15, 'sanlc2022_pct_Forest': 0.85 }
    """
    raster_source = _get_loaded_static_layer(layer_identifier)
    attr_mapping = _load_raster_attribute_table(layer_identifier)
    
    if raster_source is None:
        return {}

    try:
        raster_crs = raster_source.crs
        transformer = pyproj.Transformer.from_crs("EPSG:4326", raster_crs, always_xy=True)
        x_proj, y_proj = transformer.transform(longitude, latitude)

        bounds = raster_source.bounds
        if not (bounds.left <= x_proj <= bounds.right and bounds.bottom <= y_proj <= bounds.top):
            return {}

        point_projected = Point(x_proj, y_proj)
        buffered_area = point_projected.buffer(buffer_radius_meters)

        raster_nodata = raster_source.nodata
        masked_image, _ = mask(raster_source, [buffered_area], crop=True, nodata=raster_nodata if raster_nodata is not None else 0)
        pixel_data = masked_image[0].flatten()
        
        if raster_nodata is not None:
            valid_pixels = pixel_data[(pixel_data != raster_nodata) & (pixel_data > 0)]
        else:
            valid_pixels = pixel_data[pixel_data > 0]

        if valid_pixels.size == 0:
            return {}
        total_valid_pixels = len(valid_pixels)
        unique_vals, counts = np.unique(valid_pixels, return_counts=True)
        
        proportions = {}
        for val, count in zip(unique_vals, counts):
            class_name = attr_mapping.get(val, f"Class_{int(val)}")
            clean_name = str(class_name).replace(" ", "_").replace("/", "_").replace("-", "_")
            col_name = f"{layer_identifier}_pct_{clean_name}"
            proportions[col_name] = count / total_valid_pixels 
        return proportions

    except Exception as e:
        print(f"Error pada {layer_identifier} di {latitude},{longitude}: {e}")
        return {}

# ------ Chunked processing utility ------
def process_in_chunks(df: pd.DataFrame, process_func, chunk_size: int = 50, desc: str = "Processing") -> pd.DataFrame:
    """Process dataframe in chunks with dynamic progress tracking to avoid 20MB output limit."""
    results = []
    total_chunks = (len(df) + chunk_size - 1) // chunk_size
    for i in range(0, len(df), chunk_size):
        chunk = df.iloc[i:i + chunk_size]
        chunk_result = process_func(chunk)
        results.append(chunk_result)
        clear_output(wait=True) 
        print(f"[{desc}] Progress: Chunk {i // chunk_size + 1}/{total_chunks} done ({min(i + chunk_size, len(df))}/{len(df)} rows)")
    return pd.concat(results, ignore_index=True)

# ------ Per-API extraction functions ------
def extract_elevation_features(df: pd.DataFrame, lat_col: str = 'Latitude', lon_col: str = 'Longitude') -> pd.DataFrame:
    """Extract elevation and slope for each unique coordinate. Returns df with new spatial features."""
    coords = df[[lat_col, lon_col]].drop_duplicates().copy()
    elev_features = coords.apply(lambda r: fetch_elevation_and_slope(r[lat_col], r[lon_col]), axis=1, result_type='expand')
    coords = pd.concat([coords, elev_features], axis=1)
    return df[[lat_col, lon_col]].merge(coords, on=[lat_col, lon_col], how='left')

def extract_weather_features(df: pd.DataFrame, lat_col: str = 'Latitude', lon_col: str = 'Longitude', date_col: str = 'Sample_Date') -> pd.DataFrame:
    """Extract weather features for each unique coordinate+date. Returns df with keys + total_precipitation, average_wind_speed."""
    result = df[[lat_col, lon_col, date_col]].copy()
    weather_data = result.apply(lambda r: fetch_historical_weather(r[lat_col], r[lon_col], str(r[date_col])), axis=1)
    return pd.concat([result, pd.DataFrame(weather_data.tolist(), index=result.index)], axis=1)

def extract_soilgrids_features(df: pd.DataFrame, lat_col: str = 'Latitude', lon_col: str = 'Longitude') -> pd.DataFrame:
    """Extract multi-depth soil properties using cross-method. Returns df with all soil features."""
    coords = df[[lat_col, lon_col]].drop_duplicates().copy()
    soil_features = coords.apply(lambda r: fetch_soilgrids_cross(r[lat_col], r[lon_col]), axis=1, result_type='expand')
    coords = pd.concat([coords, soil_features], axis=1)
    return df[[lat_col, lon_col]].merge(coords, on=[lat_col, lon_col], how='left')
def extract_osm_features(df: pd.DataFrame, lat_col: str = 'Latitude', lon_col: str = 'Longitude') -> pd.DataFrame:
    """Extract density and proximity OSM features. Returns df with 9 new features."""
    coords = df[[lat_col, lon_col]].drop_duplicates().copy()
    osm_features = coords.apply(lambda r: fetch_osm_advanced(r[lat_col], r[lon_col]), axis=1, result_type='expand')
    coords = pd.concat([coords, osm_features], axis=1)
    return df[[lat_col, lon_col]].merge(coords, on=[lat_col, lon_col], how='left')

def extract_terraclimate_features(df: pd.DataFrame, lat_col: str = 'Latitude', lon_col: str = 'Longitude', date_col: str = 'Sample Date') -> pd.DataFrame:
    """
    Mengekstrak seluruh 14 fitur TerraClimate dengan mengelompokkan data berdasarkan Bulan-Tahun 
    untuk meminimalisir pemanggilan API STAC.
    """
    df = df.copy()
    df['datetime'] = pd.to_datetime(df[date_col], format='mixed', dayfirst=False)
    df['year_month'] = df['datetime'].dt.to_period('M')
    all_assets = ['pet', 'ppt', 'tmax', 'tmin', 'vap', 'srad', 'ws', 'aet', 'q', 'def', 'soil', 'swe', 'pdsi', 'vpd']
    for asset in all_assets:
        df[f'terra_{asset}'] = pd.NA
        
    unique_months = df['year_month'].unique()
    print(f"Memproses {len(unique_months)} bulan unik untuk ekstraksi TerraClimate...")
    
    for ym in unique_months:
        month_mask = df['year_month'] == ym
        points_df = df[month_mask]
        start_date = ym.to_timestamp(how='start').strftime('%Y-%m-%dT00:00:00Z')
        end_date = ym.to_timestamp(how='end').strftime('%Y-%m-%dT23:59:59Z')
        coords = [(row[lon_col], row[lat_col]) for _, row in points_df.iterrows()]
        unique_coords = list(set(coords))
        month_data = fetch_terraclimate_month(start_date, end_date, unique_coords, all_assets)
        for idx, row in points_df.iterrows():
            coord = (row[lon_col], row[lat_col])
            for asset in all_assets:
                df.at[idx, f'terra_{asset}'] = month_data[coord][asset]
    df = df.drop(columns=['datetime', 'year_month'])
    return df

def extract_landsat_features(df: pd.DataFrame, lat_col: str = 'Latitude', lon_col: str = 'Longitude', date_col: str = 'Sample_Date') -> pd.DataFrame:
    """
    Ekstrak fitur Landsat untuk setiap pasangan koordinat dan tanggal secara independen.
    Mengembalikan DataFrame dengan 6 kolom turunan satelit resolusi tinggi.
    """
    result_df = df[[lat_col, lon_col, date_col]].copy()
    landsat_features = result_df.apply(lambda r: fetch_landsat_bbox(r[lat_col], r[lon_col], str(r[date_col])), axis=1, result_type='expand')
    result_df = pd.concat([result_df, landsat_features], axis=1)
    
    return result_df

def extract_sentinel_features(df: pd.DataFrame, lat_col: str = 'Latitude', lon_col: str = 'Longitude', date_col: str = 'Sample_Date') -> pd.DataFrame:
    """
    Ekstrak fitur sentinel untuk setiap pasangan koordinat dan tanggal secara independen.
    Mengembalikan DataFrame dengan 6 kolom turunan satelit resolusi tinggi.
    """
    result_df = df[[lat_col, lon_col, date_col]].copy()
    sentinel2_features = result_df.apply(lambda r: fetch_sentinel2_bbox(r[lat_col], r[lon_col], str(r[date_col])), axis=1, result_type='expand')
    result_df = pd.concat([result_df, sentinel2_features], axis=1)
    
    return result_df

def extract_sanlc_features(df: pd.DataFrame, lat_col: str = 'Latitude', lon_col: str = 'Longitude') -> pd.DataFrame:
    """
    Ekstrak fitur SANLC menjadi kolom-kolom proporsi (0.0 - 1.0) secara dinamis.
    """
    coords = df[[lat_col, lon_col]].drop_duplicates().copy()
    # Eksekusi SANLC 2022
    print("Ekstrak Proporsi SANLC 2022 (Radius 1km)...")
    sanlc2022_features = coords.apply(lambda r: fetch_sanlc_proportions(r[lat_col], r[lon_col], "sanlc2022", 1000), axis=1, result_type='expand')
    # Eksekusi SANLC 2020
    print("Ekstrak Proporsi SANLC 2020 (Radius 1km)...")
    sanlc2020_features = coords.apply(lambda r: fetch_sanlc_proportions(r[lat_col], r[lon_col], "sanlc2020", 1000), axis=1, result_type='expand')
    coords = pd.concat([coords, sanlc2022_features, sanlc2020_features], axis=1)
    coords = coords.fillna(0.0)
    return df[[lat_col, lon_col]].merge(coords, on=[lat_col, lon_col], how='left')

def extract_worldpop_features(df: pd.DataFrame, lat_col: str = 'Latitude', lon_col: str = 'Longitude') -> pd.DataFrame:
    """
    Pembungkus DataFrame untuk WorldPop.
    """
    coords = df[[lat_col, lon_col]].drop_duplicates().copy()
    print("Ekstrak multi-fitur WorldPop (1km & 5km)...")
    wp_features = coords.apply(lambda r: fetch_worldpop_features(r[lat_col], r[lon_col]), axis=1, result_type='expand')
    coords = pd.concat([coords, wp_features], axis=1)
    return df[[lat_col, lon_col]].merge(coords, on=[lat_col, lon_col], how='left')

def extract_atlas_features_vectorized(df: pd.DataFrame, lat_col: str = 'Latitude', lon_col: str = 'Longitude') -> pd.DataFrame:
    """
    Eksekusi Spatial Join (ATLAS) dalam 1 kali komputasi terpusat (O(log M)).
    Bebas dari CRS Warning dan sangat hemat memori.
    """
    coords = df[[lat_col, lon_col]].drop_duplicates().copy()
    geometry = [Point(xy) for xy in zip(coords[lon_col], coords[lat_col])]
    points_gdf = gpd.GeoDataFrame(coords, geometry=geometry, crs="EPSG:4326")
    
    print("  -> Spatial Join dengan HydroATLAS...")
    hydro_gdf = _get_loaded_static_layer("hydroatlas")
    
    if hydro_gdf is not None and not hydro_gdf.empty:

        if points_gdf.crs != hydro_gdf.crs:
            hydro_gdf = hydro_gdf.to_crs(points_gdf.crs)
        hydro_joined = gpd.sjoin(points_gdf, hydro_gdf, how="left", predicate="intersects")
        coords['basin_upstream_area_km2'] = hydro_joined['UP_AREA'].fillna(0.0).values
        coords['basin_population'] = hydro_joined['POP'].fillna(0.0).values
        coords['basin_agriculture_pct'] = hydro_joined['AG'].fillna(0.0).values
        coords['basin_slope_deg'] = hydro_joined['SLOPE'].fillna(0.0).values
    else:
        for col in ['basin_upstream_area_km2', 'basin_population', 'basin_agriculture_pct', 'basin_slope_deg']:
            coords[col] = 0.0

    print("  -> Spatial Join dengan RiverATLAS (Nearest)...")
    river_gdf = _get_loaded_static_layer("riveratlas")
    
    if river_gdf is not None and not river_gdf.empty:
        points_meter_gdf = points_gdf.to_crs("EPSG:3857")
        river_meter_gdf = river_gdf.to_crs("EPSG:3857")

        river_joined = gpd.sjoin_nearest(
            points_meter_gdf, 
            river_meter_gdf, 
            how="left", 
            distance_col="distance_to_river_m" 
        )
        
        coords['river_avg_discharge_cms'] = river_joined['DIS_AV_CMS'].fillna(0.0).values
        coords['river_order'] = river_joined['RIV_ORD'].fillna(0).values
        coords['river_width_m'] = river_joined['RIV_WIDTH'].fillna(0.0).values
        coords['river_upstream_area_km2'] = river_joined['UP_AREA'].fillna(0.0).values
        coords['distance_to_river_m'] = river_joined['distance_to_river_m'].fillna(0.0).values
    else:
        for col in ['river_avg_discharge_cms', 'river_order', 'river_width_m', 'river_upstream_area_km2', 'distance_to_river_m']:
            coords[col] = 0.0
    return df[[lat_col, lon_col]].merge(coords, on=[lat_col, lon_col], how='left')

# ------ Snowflake Stage upload helper ------
def save_and_upload_to_stage(df: pd.DataFrame, file_name: str, session, stage_path: str = "@EXTERNAL_DATA_STAGE") -> None:
    """Save dataframe as parquet to /tmp/ and upload to Snowflake Stage."""
    local_path = f"/tmp/{file_name}"
    df.to_parquet(local_path, index=False, engine='pyarrow', compression='snappy')
    session.file.put(f"file://{local_path}", stage_path, auto_compress=False, overwrite=True)
    print(f"✓ {file_name} uploaded to {stage_path} ({len(df)} rows, {os.path.getsize(local_path) / 1024:.1f} KB)")

# ===== = base water quality CSV (train)======
session.file.get(f"{data_stage_path}/water_quality_training_dataset.csv", "/tmp/")
base_water_quality_df = pd.read_csv("/tmp/water_quality_training_dataset.csv")

# Parse Sample Date: EY dataset uses M/D/YY (American format, month first)
# Confirmed because 1/18/11 has day=18 which cannot be a month
base_water_quality_df['Sample_Date'] = pd.to_datetime(
    base_water_quality_df['Sample Date'], format='mixed', dayfirst=False
)

# Two base dataframes:
# coords_unique  -> for static/spatial APIs (Elevation, SoilGrids, OSM)
coords_unique = base_water_quality_df[['Latitude', 'Longitude']].drop_duplicates().reset_index(drop=True)

# coords_date_unique -> for time-dependent APIs (Weather)
coords_date_unique = base_water_quality_df[['Latitude', 'Longitude', 'Sample_Date']].drop_duplicates().reset_index(drop=True)

# ======== Load base water quality CSV (test coordinate) ===========
session.file.get(f"{data_stage_path}/submission_template.csv", "/tmp/")
base_sub_test_df = pd.read_csv("/tmp/submission_template.csv")

base_sub_test_df['Sample_Date'] = pd.to_datetime(
    base_sub_test_df['Sample Date'], format='mixed', dayfirst=False
)

coords_test_unique = base_sub_test_df[['Latitude', 'Longitude']].drop_duplicates().reset_index(drop=True)
coords_date_test_unique = base_sub_test_df[['Latitude', 'Longitude', 'Sample_Date']].drop_duplicates().reset_index(drop=True)

print(f"✓ Data Train loaded: {len(base_water_quality_df)} rows total")
print(f"  coords_unique (static APIs):      {len(coords_unique)} unique train coordinates")
print(f"  coords_date_unique (weather API): {len(coords_date_unique)} unique train coordinate-date pairs")

print(f"✓ Data Test loaded: {len(base_sub_test_df)} rows total")
print(f"  coords_test_unique (static APIs):      {len(coords_test_unique)} unique test coordinates")
print(f"  coords_date_test_unique (weather API): {len(coords_date_test_unique)} unique test coordinate-date pairs")

In [ ]:
import pandas as pd
from IPython.display import display

def analyze_parquet_from_stage(session, file_name, data_stage_path):
    import os
    import pandas as pd

    local_path = f"/tmp/{file_name}"
    if not os.path.exists(local_path):
        session.file.get(f"{data_stage_path}/{file_name}", "/tmp/")

    # Deteksi format file dari ekstensi
    if file_name.lower().endswith('.csv'):
        df = pd.read_csv(local_path)
    elif file_name.lower().endswith('.parquet'):
        df = pd.read_parquet(local_path)
    else:
        raise ValueError("Unsupported file format. Only .csv and .parquet are supported.")

    print(f"--- Analyze: {file_name} ---")
    print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
    print("Columns & dtypes:")
    print(df.info())

    # Missing values
    missing_pct = (df.isnull().sum() / len(df)) * 100
    missing_df = missing_pct[missing_pct > 0].reset_index()
    if missing_df.empty:
        print("\n✓ Clean data — 0% NaN")
    else:
        missing_df.columns = ['Feature', 'NaN %']
        missing_df = missing_df.sort_values('NaN %', ascending=False).reset_index(drop=True)
        print("\n⚠ Missing values:")
        print(missing_df)

    # Unique values and zero dominance
    print("\n--- Unique values & zero analysis ---")
    for col in df.columns:
        n_unique = df[col].nunique()
        output = f"Column: {col}: {n_unique} unique values"
        if pd.api.types.is_numeric_dtype(df[col]):
            zero_pct = 100 * (df[col] == 0).sum() / len(df)
            output += f", {zero_pct:.2f}% zeros"
            if zero_pct > 50:
                output += "blank"
        print(output)

    return df

def analyze_sanlc_parquet_from_stage(session, file_name: str, data_stage_path: str = "@EXTERNAL_DATA_STAGE") -> None:
    local_path = f"/tmp/{file_name}"
    session.file.get(f"{data_stage_path}/{file_name}", "/tmp/")
    df = pd.read_parquet(local_path)
    
    print(f"\n[{file_name}] Shape: {df.shape}")
    
    # 1. Missing Values
    missing_pct = (df.isnull().sum() / len(df)) * 100
    missing_df = missing_pct[missing_pct > 0].reset_index()
    if not missing_df.empty:
        missing_df.columns = ['Feature', 'NaN (%)']
        display(missing_df.sort_values(by='NaN (%)', ascending=False).reset_index(drop=True))
    else:
        print("Missing Values: 0%")
        
    # 2. Value Counts untuk kolom SANLC
    sanlc_cols = [col for col in df.columns if 'sanlc' in col.lower()]
    for col in sanlc_cols:
        val_counts = df[col].value_counts(dropna=False).reset_index()
        val_counts.columns = [f'Classes: {col}', 'Count']
        display(val_counts)
        
    # 3. Preview
    display(df)

## API 1: Elevation (OpenTopoData)

### Train 

In [ ]:
print("Extracting elevation features...")

def _elev_chunk(chunk):
    return extract_elevation_features(chunk, lat_col='Latitude', lon_col='Longitude')

elevation_df = process_in_chunks(coords_unique, _elev_chunk, chunk_size=50, desc="Elevation")

save_and_upload_to_stage(elevation_df, "train_elevation_features.parquet", session, data_stage_path)
display(elevation_df.head())

### Test 

In [ ]:
print("Extracting elevation features for data test...")

def _elev_chunk(chunk):
    return extract_elevation_features(chunk, lat_col='Latitude', lon_col='Longitude')

elevation_test_df = process_in_chunks(coords_test_unique, _elev_chunk, chunk_size=50, desc="Elevation Test")
save_and_upload_to_stage(elevation_test_df, "test_elevation_features.parquet", session, data_stage_path)

## API 2: Historical Weather 7-day Lag (Open-Meteo)

### Train

In [ ]:
print("Extracting weather features...")
weather_train_df = extract_weather_vectorized(df=coords_date_unique, lat_col='Latitude', lon_col='Longitude', date_col='Sample Date')
save_and_upload_to_stage(weather_train_df, "train_weather_features.parquet", session, data_stage_path)
display(weather_df.head())

### Test

In [ ]:
print("Extracting weather features ...")
weather_test_df = extract_weather_vectorized( df=coords_date_test_unique, lat_col='Latitude', lon_col='Longitude', date_col='Sample Date')
save_and_upload_to_stage(weather_df, "train_weather_features.parquet", session, data_stage_path)
display(weather_df.head())

## API 3: Soil Properties (ISRIC SoilGrids)

### Train

In [ ]:
print("Extracting SoilGrids features (6 properties per coordinate)...")

def _soil_chunk(chunk):
    return extract_soilgrids_features(chunk, lat_col='Latitude', lon_col='Longitude')

# chunk_size=20 because SoilGrids makes 6 API calls per coordinate
soilgrids_df = process_in_chunks(coords_unique, _soil_chunk, chunk_size=20, desc="SoilGrids")

save_and_upload_to_stage(soilgrids_df, "train_soilgrids_features.parquet", session, data_stage_path)
display(soilgrids_df.head())

### Test

In [ ]:
print("Extracting SoilGrids features (6 properties per coordinate) for data test...")

def _soil_chunk(chunk):
    return extract_soilgrids_features(chunk, lat_col='Latitude', lon_col='Longitude')

# chunk_size=20 because SoilGrids makes 6 API calls per coordinate
soilgrids_test_df = process_in_chunks(coords_test_unique, _soil_chunk, chunk_size=20, desc="SoilGrids")

save_and_upload_to_stage(soilgrids_test_df, "test_soilgrids_features.parquet", session, data_stage_path)
display(soilgrids_test_df)

## API 4: OSM Pollution Proximity (Overpass)

### Train

In [ ]:
print("Extracting OSM pollution proximity features...")
def _osm_chunk(chunk):
    return extract_osm_features(chunk, lat_col='Latitude', lon_col='Longitude')
osm_df = process_in_chunks(coords_unique, _osm_chunk, chunk_size=10, desc="OSM")
save_and_upload_to_stage(osm_df, "train_osm_features.parquet", session, data_stage_path)
display(osm_df.head())

### Test

In [ ]:
print("Extracting OSM pollution proximity features for data test...")
def _osm_chunk(chunk): 
    return extract_osm_features(chunk, lat_col='Latitude', lon_col='Longitude')
osm_test_df = process_in_chunks(coords_test_unique, _osm_chunk, chunk_size=10, desc="OSM")
save_and_upload_to_stage(osm_test_df, "test_osm_features.parquet", session, data_stage_path)
display(osm_test_df.head())

## API 5: Landsat Planetarium

### Train

In [ ]:
print("Memulai Ekstraksi Landsat (Smart BBox & Cloud Masking) untuk TRAIN data...")

def _landsat_train_chunk(chunk):
    return extract_landsat_features(chunk, lat_col='Latitude', lon_col='Longitude', date_col='Sample_Date')
landsat_train_df = process_in_chunks(coords_date_unique, _landsat_train_chunk, chunk_size=25, desc="Landsat Train")
save_and_upload_to_stage(landsat_train_df, "train_landsat.parquet", session, data_stage_path)

Test

### Test

In [ ]:
print("Memulai Ekstraksi Landsat (Smart BBox & Cloud Masking) untuk TEST data...")

def _landsat_test_chunk(chunk):
    return extract_landsat_features(chunk, lat_col='Latitude', lon_col='Longitude', date_col='Sample_Date')
landsat_test_df = process_in_chunks(coords_date_test_unique, _landsat_test_chunk, chunk_size=25, desc="Landsat Test")
save_and_upload_to_stage(landsat_test_df, "test_landsat.parquet", session, data_stage_path)

## API 6: TerraClimate

### Train

In [ ]:
print("Extracting full TerraClimate features (14 variables)...")
terraclimate_df = extract_terraclimate_features(coords_date_unique)

save_and_upload_to_stage(terraclimate_df, "train_terraclimate.parquet", session, data_stage_path)
display(terraclimate_df.head())

### Test

In [ ]:
print("Extracting full TerraClimate features (14 variables)...")
terraclimate_test_df = extract_terraclimate_features(coords_date_test_unique)

save_and_upload_to_stage(terraclimate_test_df, "test_terraclimate.parquet", session, data_stage_path)
display(terraclimate_test_df.head())

## API 8: Sentinel-2

### Train

In [ ]:
print("Memulai Ekstraksi Sentinel-2 (Smart BBox & Cloud Masking) untuk TRAIN data...")

def _sentinel2_train_chunk(chunk):
    return extract_sentinel_features(chunk, lat_col='Latitude', lon_col='Longitude', date_col='Sample_Date')
sentinel_train_df = process_in_chunks(coords_date_unique, _sentinel2_train_chunk, chunk_size=25, desc="Sentinel2 Train")
save_and_upload_to_stage(sentinel_train_df, "train_sentinel.parquet", session, data_stage_path)

### Test

In [ ]:
print("Memulai Ekstraksi Sentinel-2 (Smart BBox & Cloud Masking) untuk TEST data...")

def _sentinel2_test_chunk(chunk):
    return extract_sentinel_features(chunk, lat_col='Latitude', lon_col='Longitude', date_col='Sample_Date')
sentinel_test_df = process_in_chunks(coords_date_unique, _sentinel2_test_chunk, chunk_size=25, desc="Sentinel2 Test")
save_and_upload_to_stage(sentinel_test_df, "test_sentinel.parquet", session, data_stage_path)

## Note: Static Raster Data

Static raster data — **WorldPop**, **SANLC 2022/2020**, **HydroATLAS**, and **RiverATLAS** — are **NOT** extracted via API in this notebook.

These datasets require large raster/vector files that must be downloaded from the Snowflake Stage to `/tmp/` first (see `01-setup-static-data.ipynb`). They are handled separately to keep this notebook focused on API calls only.

The merge of all feature parquet files (API + static raster) happens in **notebook 03 (EDA & Profiling)**.

## Verification: List All Uploaded Files

In [ ]:
# Verify all parquet files are present in the stage
print("Files in EXTERNAL_DATA_STAGE:")
session.sql("LIST @EXTERNAL_DATA_STAGE").show()

### Calling /temp Statci data

In [ ]:
print("Mengunduh file statis dari Stage ke /tmp/...")

static_files = [
    "zaf_pop_2025_CN_100m_R2025A_v1.tif",
    "BasinATLAS_v10_lev12.parquet",
    "RiverATLAS_Data_v10.parquet"
]
for file in static_files:
    session.file.get(f"{data_stage_path}/{file}", "/tmp/")
    print(f"  ✓ {file} terunduh.")

## Extraction Data Worldpop 

In [ ]:
print("\n[TRAIN] Mengekstrak WorldPop...")
def _worldpop_chunk(chunk):
    coords = chunk[['Latitude', 'Longitude']].copy()
    coords['worldpop_mean_1km'] = coords.apply(lambda r: extract_worldpop_features(r['Latitude'], r['Longitude'], "worldpop", 1000), axis=1)
    return coords

train_worldpop_df = process_in_chunks(coords_unique, _worldpop_chunk, chunk_size=100, desc="WorldPop Train")
save_and_upload_to_stage(train_worldpop_df, "train_worldpop_features.parquet", session, data_stage_path)
display(train_worldpop_df)

In [ ]:
print("\n[TEST] Mengekstrak WorldPop...")
test_worldpop_df = process_in_chunks(coords_test_unique, _worldpop_chunk, chunk_size=100, desc="WorldPop Test")
save_and_upload_to_stage(test_worldpop_df, "test_worldpop_features.parquet", session, data_stage_path)
display(test_worldpop_df)

## Extraction SANLC

### Train

In [ ]:
print("Memulai Ekstraksi SANLC (Proporsi Kelas 2020 & 2022) untuk TRAIN data...")

def _sanlc_train_chunk(chunk):
    return extract_sanlc_features(chunk, lat_col='Latitude', lon_col='Longitude')
sanlc_train_df = process_in_chunks_with_checkpoint(df=coords_unique, process_func=_sanlc_train_chunk, chunk_size=50, desc="SANLC Train", checkpoint_file="/tmp/sanlc_train_checkpoint.parquet")
save_and_upload_to_stage(sanlc_train_df, "train_sanlc_features.parquet", session, data_stage_path)

### Test

In [ ]:
print("Memulai Ekstraksi SANLC (Proporsi Kelas 2020 & 2022) untuk TEST data...")

def _sanlc_test_chunk(chunk):
    return extract_sanlc_features(chunk, lat_col='Latitude', lon_col='Longitude')
sanlc_test_df = process_in_chunks_with_checkpoint(df=coords_test_unique, process_func=_sanlc_test_chunk, chunk_size=50, desc="SANLC Test",checkpoint_file="/tmp/sanlc_test_checkpoint.parquet")
save_and_upload_to_stage(sanlc_test_df, "test_sanlc_features.parquet", session, data_stage_path)

### Extraction RiverATLAS & BasinATLAST (NEED HIGH COMPUTE)

In [ ]:
print("\n[TRAIN] Mengekstrak Hidrologi (Basin & River) secara Vektor...")
train_hydro_df = extract_atlas_features_vectorized(coords_unique)
save_and_upload_to_stage(train_hydro_df, "train_atlas.parquet", session, data_stage_path)

In [ ]:
print("\n[TEST] Mengekstrak Hidrologi (Basin & River) secara Vektor...")
test_hydro_df = extract_atlas_features_vectorized(coords_test_unique)
save_and_upload_to_stage(test_hydro_df, "test_atlas.parquet", session, data_stage_path)

## Quick Sanity Check's

In [ ]:
analyze_parquet_from_stage(session, "elevation_features.parquet", data_stage_path)

In [ ]:
analyze_parquet_from_stage(session, "weather_features.parquet", data_stage_path)

In [ ]:
analyze_parquet_from_stage(session, "soilgrids_features.parquet", data_stage_path)

In [ ]:
analyze_parquet_from_stage(session, "static_features.parquet", data_stage_path)

In [ ]:
analyze_parquet_from_stage(session, "train_hydro_features.parquet", data_stage_path)

In [ ]:
def print_unique_sanlc_classes(session, file_name: str, stage_path: str = "@EXTERNAL_DATA_STAGE") -> None:
    local_path = f"/tmp/{file_name}"
    session.file.get(f"{stage_path}/{file_name}", "/tmp/")
    df = pd.read_parquet(local_path)
    
    print(f"\n======================================")
    print(f"File: {file_name}")
    print(f"======================================")
    
    sanlc_cols = [col for col in df.columns if 'sanlc' in col.lower()]
    
    if not sanlc_cols:
        print("Tidak ada kolom SANLC ditemukan.")
        return
        
    for col in sanlc_cols:
        # Mengambil nilai unik dan membuang NaN jika ada
        unique_values = df[col].dropna().unique().tolist()
        
        # Mengurutkan secara alfabetis agar mudah dibaca dan dipetakan
        unique_values.sort()
        
        print(f"\nKolom: {col}")
        print(f"Total Kelas Unik: {len(unique_values)}")
        print("Daftar Nilai (Bisa langsung di-copy):")
        print(unique_values)

In [ ]:
print_unique_sanlc_classes(session, "train_sanlc_features.parquet", data_stage_path)

In [ ]:
print_unique_sanlc_classes(session, "test_sanlc_features.parquet", data_stage_path)

In [ ]:
analyze_parquet_from_stage(session, "train_worldpop_features.parquet", data_stage_path)

In [ ]:
analyze_parquet_from_stage(session,"landsat_features_training.csv", data_stage_path)

In [ ]:
analyze_parquet_from_stage(session, "terraclimate_features_training.csv", data_stage_path)

In [ ]:
analyze_parquet_from_stage(session, "water_quality_training_dataset.csv", data_stage_path)

In [ ]:
analyze_sanlc_parquet_from_stage(session, "test_sanlc_features.parquet", data_stage_path)

## Naming Train Dataset

In [ ]:
%%sql -r dataframe_1
ALTER STAGE EXTERNAL_DATA_STAGE RENAME 'elevation_features.parquet' TO 'train_elevation_features.parquet';
ALTER STAGE EXTERNAL_DATA_STAGE RENAME 'weather_features.parquet' TO 'train_weather_features.parquet';
ALTER STAGE EXTERNAL_DATA_STAGE RENAME 'soilgrids_features.parquet' TO 'train_soilgrids_features.parquet';
ALTER STAGE EXTERNAL_DATA_STAGE RENAME 'weather_features.parquet' TO 'train_weather_features.parquet';

## Extraction Data SANLC

In [ ]:
files_to_download = [
    "SA_NLC_2020_ALBERS.tif",
    "SA_NLC_2020_ALBERS.tif.vat.dbf",
    "SA_NLC_2022_ALBERS.tif",
    "SA_NLC_2022_ALBERS.tif.vat.dbf"
]

for file_name in files_to_download:
    session.file.get(f"{data_stage_path}/{file_name}", "/tmp/")
    print(f"  ✓ {file_name} berhasil dimuat.")

In [ ]:
print("\n--- Mengekstrak SANLC untuk Data TRAIN ---")

def _sanlc_chunk(chunk):
    return extract_sanlc_features(chunk, lat_col='Latitude', lon_col='Longitude')

train_sanlc_df = process_in_chunks(coords_unique, _sanlc_chunk, chunk_size=200, desc="SANLC Train")
# save_and_upload_to_stage(train_sanlc_df, "train_sanlc_features.parquet", session, data_stage_path)
display(train_sanlc_df.head())

In [ ]:
print("\nMulai mengekstrak SANLC untuk Data TEST...")
test_sanlc_df = process_in_chunks(coords_test_unique, _sanlc_chunk, chunk_size=200, desc="SANLC Test")

save_and_upload_to_stage(test_sanlc_df, "test_sanlc_features.parquet", session, data_stage_path)
display(test_sanlc_df.head())

In [ ]:
# Test dengan koordinat yang bermasalah
test_coords = [
    (-28.760833, 17.730278),   # Orange River muara
    (-26.861111, 28.884722),   # Dekat Johannesburg
    (-26.450000, 28.085833),   # Gauteng area
]

for lat, lon in test_coords:
    print(f"\n{'='*60}")
    print(f"Testing ({lat}, {lon})")
    result = fetch_mapped_sanlc_class_debug(lat, lon, "sanlc2022", 1000)
    print(f"Result: {result}")

## OSM Extraction

In [ ]:
import os
import requests
import zipfile
from snowflake.snowpark.context import get_active_session

session = get_active_session()
data_stage_path = "@SNOWFLAKE_LEARNING_DB.PUBLIC.EXTERNAL_DATA_STAGE"
OSM_URL = "https://download.geofabrik.de/africa/south-africa-latest-free.gpkg.zip"
FILE_NAME_ZIP = "south-africa-osm.gpkg.zip"
LOCAL_ZIP = f"/tmp/{FILE_NAME_ZIP}"

def prepare_osm_data():
    # 1. Cek Stage
    files_in_stage = session.sql(f"LIST {data_stage_path}").collect()
    file_exists_in_stage = any(FILE_NAME_ZIP in file['name'] for file in files_in_stage)

    if not os.path.exists(LOCAL_ZIP):
        if file_exists_in_stage:
            print("Mengambil ZIP dari Stage...")
            session.file.get(f"{data_stage_path}/{FILE_NAME_ZIP}", "/tmp/")
        else:
            print("Mengunduh ZIP dari Geofabrik...")
            response = requests.get(OSM_URL, stream=True)
            with open(LOCAL_ZIP, 'wb') as f:
                for chunk in response.iter_content(chunk_size=1024*1024):
                    f.write(chunk)
            # Upload ke stage untuk backup
            session.file.put(f"file://{LOCAL_ZIP}", data_stage_path, auto_compress=False, overwrite=True)

    # 2. Ekstraksi dan Verifikasi
    print("Mengekstrak file...")
    with zipfile.ZipFile(LOCAL_ZIP, 'r') as zip_ref:
        # List isi zip untuk debugging
        zip_contents = zip_ref.namelist()
        print(f"Isi dalam ZIP: {zip_contents}")
        zip_ref.extractall("/tmp/")
    
    # 3. Cari file .gpkg yang sebenarnya di /tmp
    extracted_files = os.listdir("/tmp")
    gpkg_files = [f for f in extracted_files if f.endswith('.gpkg')]
    
    if gpkg_files:
        actual_path = f"/tmp/{gpkg_files[0]}"
        print(f"✓ File ditemukan: {actual_path}")
        return actual_path
    else:
        raise FileNotFoundError("Tidak ada file .gpkg yang ditemukan di /tmp setelah ekstraksi.")

# Jalankan dan simpan path aslinya
ACTUAL_GPKG_PATH = prepare_osm_data()

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

def get_nearest_distance(stations_gdf, target_gdf):
    """Menghitung jarak terdekat dari stasiun ke target (Titik/Garis/Poligon)."""
    if target_gdf.empty:
        return pd.Series([np.nan] * len(stations_gdf), index=stations_gdf.index)
    
    # Ambil centroid dari target untuk pendekatan titik terdekat
    target_points = target_gdf.geometry.centroid
    tree = cKDTree(np.stack([target_points.x, target_points.y], axis=1))
    
    stations_points = np.stack([stations_gdf.geometry.x, stations_gdf.geometry.y], axis=1)
    distances, _ = tree.query(stations_points)
    
    return distances

def calculate_polygon_area_percentage(stations_gdf, target_polygons, radius):
    """Menghitung persentase luas area poligon di dalam buffer radius tertentu."""
    if target_polygons.empty:
        return pd.Series([0.0] * len(stations_gdf), index=stations_gdf.index)
    
    buffers = stations_gdf.geometry.buffer(radius)
    buffer_area = np.pi * (radius ** 2)
    
    percentages = []
    # Menggunakan spatial index bawaan Geopandas secara implisit melalui iterasi yang difilter
    for geom in buffers:
        # Filter awal: cari poligon yang menyentuh buffer ini saja
        possible_matches = target_polygons[target_polygons.intersects(geom)]
        
        if possible_matches.empty:
            percentages.append(0.0)
        else:
            # Hitung irisan pasti (intersection) dan total luasnya
            intersected_area = possible_matches.intersection(geom).area.sum()
            perc = (intersected_area / buffer_area) * 100.0
            percentages.append(perc)
            
    return pd.Series(percentages, index=stations_gdf.index)

def calculate_line_length(stations_gdf, target_lines, radius):
    """Menghitung total panjang garis (meter) di dalam buffer radius tertentu."""
    if target_lines.empty:
        return pd.Series([0.0] * len(stations_gdf), index=stations_gdf.index)
    
    buffers = stations_gdf.geometry.buffer(radius)
    lengths = []
    
    for geom in buffers:
        possible_matches = target_lines[target_lines.intersects(geom)]
        if possible_matches.empty:
            lengths.append(0.0)
        else:
            # Hitung irisan garis dan total panjangnya
            intersected_length = possible_matches.intersection(geom).length.sum()
            lengths.append(intersected_length)
            
    return pd.Series(lengths, index=stations_gdf.index)

In [ ]:
def extract_osm_environmental_features(base_df: pd.DataFrame, gpkg_path: str) -> pd.DataFrame:
    print(f"Membaca layer OSM dari: {gpkg_path}")
    
    # 1. Load Layers
    landuse_gdf = gpd.read_file(gpkg_path, layer='gis_osm_landuse_a_free')
    roads_gdf = gpd.read_file(gpkg_path, layer='gis_osm_roads_free')
    pois_pt_gdf = gpd.read_file(gpkg_path, layer='gis_osm_pois_free')
    pois_poly_gdf = gpd.read_file(gpkg_path, layer='gis_osm_pois_a_free')
    
    # 2. Filter Kategori Target
    mines = landuse_gdf[landuse_gdf['fclass'].isin(['quarry', 'mine'])]
    farms = landuse_gdf[landuse_gdf['fclass'].isin(['farmland', 'farmyard', 'orchard', 'vineyard', 'meadow'])]
    roads = roads_gdf[roads_gdf['fclass'].isin(['primary', 'secondary', 'tertiary', 'trunk', 'motorway'])]
    
    wastewater_pt = pois_pt_gdf[pois_pt_gdf['fclass'] == 'wastewater_plant']
    wastewater_poly = pois_poly_gdf[pois_poly_gdf['fclass'] == 'wastewater_plant']
    
    # 3. Standardisasi Proyeksi Ruang (EPSG:32734 - Meter)
    target_crs = "EPSG:32734"
    
    mines_proj = mines.to_crs(target_crs)
    farms_proj = farms.to_crs(target_crs)
    roads_proj = roads.to_crs(target_crs)
    
    # Pemrosesan Limbah yang Aman (Proyeksi -> Centroid -> Gabung)
    wastewater_pt_proj = wastewater_pt.to_crs(target_crs)
    wastewater_poly_proj = wastewater_poly.to_crs(target_crs)
    
    if not wastewater_poly_proj.empty:
        wastewater_poly_centroids = wastewater_poly_proj.copy()
        wastewater_poly_centroids['geometry'] = wastewater_poly_centroids.geometry.centroid
    else:
        wastewater_poly_centroids = gpd.GeoDataFrame(geometry=[], crs=target_crs)
        
    wastewater_combined_proj = pd.concat([wastewater_pt_proj, wastewater_poly_centroids])
    
    # Proyeksi Titik Stasiun
    stations_proj = gpd.GeoDataFrame(
        base_df, 
        geometry=gpd.points_from_xy(base_df['Longitude'], base_df['Latitude']),
        crs="EPSG:4326"
    ).to_crs(target_crs)
    
    # 4. Eksekusi Perhitungan Metrik Spasial
    print("Menghitung fitur Tambang (Jarak & Persentase Area)...")
    stations_proj['osm_dist_nearest_mine'] = get_nearest_distance(stations_proj, mines_proj)
    stations_proj['osm_mine_area_perc_1km'] = calculate_polygon_area_percentage(stations_proj, mines_proj, 1000)
    stations_proj['osm_mine_area_perc_5km'] = calculate_polygon_area_percentage(stations_proj, mines_proj, 5000)
    
    print("Menghitung fitur Pertanian (Jarak & Persentase Area)...")
    stations_proj['osm_dist_nearest_farm'] = get_nearest_distance(stations_proj, farms_proj)
    stations_proj['osm_farm_area_perc_1km'] = calculate_polygon_area_percentage(stations_proj, farms_proj, 1000)
    stations_proj['osm_farm_area_perc_5km'] = calculate_polygon_area_percentage(stations_proj, farms_proj, 5000)
    
    print("Menghitung fitur Jalan Raya (Panjang Permukaan Kedap Air)...")
    stations_proj['osm_dist_nearest_road'] = get_nearest_distance(stations_proj, roads_proj)
    stations_proj['osm_road_length_m_1km'] = calculate_line_length(stations_proj, roads_proj, 1000)
    stations_proj['osm_road_length_m_5km'] = calculate_line_length(stations_proj, roads_proj, 5000)
    
    print("Menghitung fitur Pengolahan Limbah (Jarak & Kepadatan Titik)...")
    stations_proj['osm_dist_nearest_wastewater'] = get_nearest_distance(stations_proj, wastewater_combined_proj)
    
    stations_proj['osm_wastewater_count_1km'] = stations_proj.geometry.buffer(1000).apply(
        lambda x: wastewater_combined_proj.intersects(x).sum() if not wastewater_combined_proj.empty else 0
    )
    stations_proj['osm_wastewater_count_5km'] = stations_proj.geometry.buffer(5000).apply(
        lambda x: wastewater_combined_proj.intersects(x).sum() if not wastewater_combined_proj.empty else 0
    )
    
    return pd.DataFrame(stations_proj.drop(columns='geometry'))

In [ ]:
def save_osm_to_stage(dataframe, filename):
    local_path = f"/tmp/{filename}"
    dataframe.to_parquet(local_path, index=False)
    session.file.put(f"file://{local_path}", data_stage_path, auto_compress=False, overwrite=True)
    print(f"✓ Berhasil disimpan ke Stage: {filename}")

print("Memulai Ekstraksi OSM: Train Dataset")
osm_train_features = extract_osm_environmental_features(coords_unique, ACTUAL_GPKG_PATH)
save_osm_to_stage(osm_train_features, "train_osm.parquet")

print("\nMemulai Ekstraksi OSM: Test Dataset")
osm_test_features = extract_osm_environmental_features(coords_test_unique, ACTUAL_GPKG_PATH)
save_osm_to_stage(osm_test_features, "test_osm.parquet")

display(osm_train_features.head())